In [42]:
# making imports

import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout

RANDOM_STATE = 42
WINDOW_SIZE = 10

### Window Size Choice
A window size of 10 days was chosen so the LSTM can capture short-term momentum patterns while avoiding excessive noise.

### Train/Test Split Strategy
The split is chronological:
- First 80% → train
- Last 20% → test

This is required in time-series because future values must never leak into training.

### Why Random Split Is Wrong
Random split leaks future patterns into training, producing unrealistically good test performance and invalidating the evaluation.

In [43]:
stock_df = pd.read_csv("stock_prices.csv")
stock_df.head()

,date,ticker,open,high,low,close,volume,returns_pct
0,2022-01-03,RELIANCE,2476.84,2496.70,2422.91,2459.80,4940428,-1.6079
1,2022-01-04,RELIANCE,2443.34,2497.22,2423.41,2460.31,6310409,0.0207
2,2022-01-05,RELIANCE,2467.38,2481.76,2408.41,2445.09,2920856,-0.6189
3,2022-01-06,RELIANCE,2450.52,2472.16,2399.09,2435.63,2458621,-0.3868
4,2022-01-07,RELIANCE,2415.56,2439.90,2367.79,2403.85,4011166,-1.3049


In [44]:
stock_name = stock_df['ticker'].unique()[0]
stock_one = stock_df[stock_df['ticker'] == stock_name].copy()
stock_one = stock_one.sort_values("date")

prices = stock_one["close"].values.reshape(-1,1)

scaler = MinMaxScaler()
scaled_prices = scaler.fit_transform(prices)

In [45]:
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(window_size, len(data)):
        X.append(data[i-window_size:i])
        y.append(data[i])
    return np.array(X), np.array(y)

X_stock, y_stock = create_sequences(scaled_prices, WINDOW_SIZE)

split_idx = int(len(X_stock)*0.8)

X_train_stock = X_stock[:split_idx]
X_test_stock = X_stock[split_idx:]
y_train_stock = y_stock[:split_idx]
y_test_stock = y_stock[split_idx:]

In [46]:
stock_model = Sequential([
    LSTM(64, input_shape=(WINDOW_SIZE,1)),
    Dense(1)
])

stock_model.compile(optimizer='adam', loss='mse')

history = stock_model.fit(
    X_train_stock, y_train_stock,
    epochs=10,
    batch_size=32,
    validation_split=0.1,
    verbose=0
)

In [47]:
pred_scaled = stock_model.predict(X_test_stock)
pred = scaler.inverse_transform(pred_scaled)
actual = scaler.inverse_transform(y_test_stock)

mae = mean_absolute_error(actual, pred)
print("Stock MAE:", mae)

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step
Stock MAE: 109.77661871832768


MAE is the most meaningful metric for trading because it measures the average absolute price error.

For deployment, MAE should be significantly lower than average daily volatility; otherwise predictions do not create actionable trading signals.

In [48]:
import kagglehub

chat_path = kagglehub.dataset_download("bitext/bitext-gen-ai-chatbot-customer-support-dataset")
print(chat_path)

C:\Users\SHREE\.cache\kagglehub\datasets\bitext\bitext-gen-ai-chatbot-customer-support-dataset\versions\1


In [49]:
files = os.listdir(chat_path)
files

['Bitext_Sample_Customer_Support_Training_Dataset_27K_responses-v11.csv']

In [50]:
chat_df = pd.read_csv("chat_logs.csv")
chat_df.head()

,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [51]:
chat_df = pd.read_csv("chat_logs.csv")

chat_df["flags"] = pd.to_numeric(chat_df["flags"], errors="coerce").fillna(0)
chat_df["churn_risk"] = (chat_df["flags"] > 0).astype(int)

chat_df["instruction_len"] = chat_df["instruction"].astype(str).apply(len)
chat_df["response_len"] = chat_df["response"].astype(str).apply(len)

print(chat_df["churn_risk"].value_counts())
chat_df.head()

churn_risk
0    26872
Name: count, dtype: int64


,flags,instruction,category,intent,response,churn_risk,instruction_len,response_len
0,0.0,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,0,48,212
1,0.0,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...,0,58,279
2,0.0,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,0,47,1343
3,0.0,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,0,42,1104
4,0.0,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...,0,60,1222


In [52]:
high_risk_intents = [
    "complaint",
    "refund_request",
    "cancellation",
    "billing_issue",
    "escalation"
]

chat_df["churn_risk"] = chat_df["intent"].apply(
    lambda x: 1 if x in high_risk_intents else 0
)

print(chat_df["churn_risk"].value_counts())

churn_risk
0    25872
1     1000
Name: count, dtype: int64


In [53]:
print(chat_df["intent"].unique())

['cancel_order' 'change_order' 'change_shipping_address'
 'check_cancellation_fee' 'check_invoice' 'check_payment_methods'
 'check_refund_policy' 'complaint' 'contact_customer_service'
 'contact_human_agent' 'create_account' 'delete_account'
 'delivery_options' 'delivery_period' 'edit_account' 'get_invoice'
 'get_refund' 'newsletter_subscription' 'payment_issue' 'place_order'
 'recover_password' 'registration_problems' 'review'
 'set_up_shipping_address' 'switch_account' 'track_order' 'track_refund']


In [54]:
chat_df["intent"].value_counts().head(20)

intent
edit_account                1000
switch_account              1000
check_invoice               1000
complaint                   1000
contact_customer_service    1000
delivery_period              999
registration_problems        999
check_payment_methods        999
contact_human_agent          999
payment_issue                999
newsletter_subscription      999
get_invoice                  999
place_order                  998
cancel_order                 998
track_refund                 998
change_order                 997
get_refund                   997
create_account               997
check_refund_policy          997
review                       997
Name: count, dtype: int64

In [55]:
from sklearn.preprocessing import LabelEncoder

category_encoder = LabelEncoder()
intent_encoder = LabelEncoder()

chat_df["category_encoded"] = category_encoder.fit_transform(chat_df["category"])
chat_df["intent_encoded"] = intent_encoder.fit_transform(chat_df["intent"])

In [56]:
chat_df["instruction_len"] = chat_df["instruction"].astype(str).apply(len)
chat_df["response_len"] = chat_df["response"].astype(str).apply(len)

features = [
    "instruction_len",
    "response_len",
    "category_encoded",
    "intent_encoded"
]

X = chat_df[features]
y = chat_df["churn_risk"]

print(X.head())
print(y.value_counts())

   instruction_len  response_len  category_encoded  intent_encoded
0               48           212                 6               0
1               58           279                 6               0
2               47          1343                 6               0
3               42          1104                 6               0
4               60          1222                 6               0
churn_risk
0    25872
1     1000
Name: count, dtype: int64


In [57]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)

(21497, 4) (5375, 4)


In [58]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [59]:
from sklearn.metrics import roc_auc_score, classification_report

pred_probs = rf_model.predict_proba(X_test)[:,1]
preds = rf_model.predict(X_test)

auc = roc_auc_score(y_test, pred_probs)

print("ROC-AUC:", auc)
print(classification_report(y_test, preds))

ROC-AUC: 1.0
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5175
           1       1.00      1.00      1.00       200

    accuracy                           1.00      5375
   macro avg       1.00      1.00      1.00      5375
weighted avg       1.00      1.00      1.00      5375



ROC-AUC is used because churn prediction is an imbalanced classification problem.

AUC measures ranking quality across thresholds, making it more informative than raw accuracy.

A sequence model was not selected because the most predictive signals in this dataset are structured interaction attributes (intent, category, message lengths), not temporal message order.

Therefore a tabular ensemble model is expected to perform comparably with lower complexity.

In [60]:
chat_df["risk_score"] = rf_model.predict_proba(X)[:,1]

risk_ranked = chat_df.sort_values("risk_score", ascending=False)

risk_ranked[
    ["instruction", "category", "intent", "risk_score"]
].head(10)

,instruction,category,intent,risk_score
6968,I don't know how I can make a claim against yo...,FEEDBACK,complaint,1.0
7571,help to make a complaint against your business,FEEDBACK,complaint,1.0
7573,I don't know what I need to do to file a custo...,FEEDBACK,complaint,1.0
7574,file customerr claim,FEEDBACK,complaint,1.0
7575,how to make a claim against your business?,FEEDBACK,complaint,1.0
7576,help lodging a consumer reclamation against yo...,FEEDBACK,complaint,1.0
7577,I don't know what I need to do to make a custo...,FEEDBACK,complaint,1.0
7578,help me lodge a consumer claim against ur company,FEEDBACK,complaint,1.0
7579,where to lodge a customre claim against ur org...,FEEDBACK,complaint,1.0
7580,I have to make a reclamation,FEEDBACK,complaint,1.0


In [61]:
false_positive_cost = 2
false_negative_cost = 10

threshold = false_positive_cost / (false_positive_cost + false_negative_cost)

print("Cost-effective threshold:", threshold)

Cost-effective threshold: 0.16666666666666666


In [62]:
high_risk_cases = risk_ranked[risk_ranked["risk_score"] >= threshold]

print("Cases needing intervention:", len(high_risk_cases))

Cases needing intervention: 1000


Missing a high-risk churn case is assumed to cost five times more than unnecessary outreach.

Therefore intervention is triggered when predicted churn probability exceeds 0.167.

In [63]:
baseline_preds = []

for seq in X_test_stock:
    baseline_preds.append(np.mean(seq))

baseline_preds = np.array(baseline_preds).reshape(-1,1)

In [64]:
from sklearn.metrics import mean_absolute_error

baseline_preds_inv = scaler.inverse_transform(baseline_preds)
actual_inv = scaler.inverse_transform(y_test_stock.reshape(-1,1))

baseline_mae = mean_absolute_error(actual_inv, baseline_preds_inv)

print("Baseline MAE:", baseline_mae)
print("LSTM MAE:", mae)

Baseline MAE: 46.02667567567566
LSTM MAE: 109.77661871832768


The moving-average baseline tests whether LSTM complexity adds predictive value.

If the baseline performs similarly, the stock series likely contains strong short-term autocorrelation but limited nonlinear structure.